# Inflating a thick elliptical cylinder

In [ ]:
from typing import Literal
from pathlib import Path
from mpi4py import MPI
import dolfinx
import logging
from dolfinx import log
import ufl
import numpy as np
from scipy.integrate import solve_ivp
import adios4dolfinx
import pulse
import cardiac_geometries
import cardiac_geometries.geometry
import pyvista

In [ ]:
comm = MPI.COMM_WORLD
target_pressure = 10_000  # Pa
char_length = 0.003
width = 0.01
r_inner_x = 0.025
r_inner_y = 0.015
r_outer_x = r_inner_x + width
r_outer_y = r_inner_y + width
height = 0.08
outdir = Path("results_thick_elliptical_cylinder")
geodir = outdir / "geometry"
fiber_angle = 0.0
fiber_space = "DG_1"

In [ ]:
if not geodir.exists():
    cardiac_geometries.mesh.cylinder_elliptical(
        outdir=geodir,
        create_fibers=True,
        fiber_space="DG_1",
        r_inner_x=r_inner_x,
        r_inner_y=r_inner_y,
        r_outer_x=r_outer_x,
        r_outer_y=r_outer_y,
        height=height,
        char_length=char_length,
        comm=comm,
        fiber_angle_epi=-fiber_angle,
        fiber_angle_endo=fiber_angle,
    )

In [ ]:
# If the folder already exist, then we just load the geometry
geo = cardiac_geometries.geometry.Geometry.from_folder(
    comm=comm,
    folder=geodir,
)
geo.markers

In [ ]:
set(geo.ffun.values)

In [ ]:
vtk_mesh = dolfinx.plot.vtk_mesh(geo.mesh, geo.mesh.topology.dim)
grid = pyvista.UnstructuredGrid(*vtk_mesh)
plotter = pyvista.Plotter()
plotter.add_mesh(grid, show_edges=True)
plotter.view_xy()
plotter.show_bounds()
if not pyvista.OFF_SCREEN:
    plotter.show()
else:
    plotter.screenshot(outdir / "cylinder_mesh.png")

## Fiber: Circumferential direction

In [ ]:
assert geo.f0 is not None
topology, cell_types, geometry = dolfinx.plot.vtk_mesh(geo.f0.function_space)
values = np.zeros((geometry.shape[0], 3), dtype=np.float64)
values[:, : len(geo.f0)] = geo.f0.x.array.real.reshape((geometry.shape[0], len(geo.f0)))
function_grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)
function_grid["u"] = values
glyphs = function_grid.glyph(orient="u", factor=char_length)
grid = pyvista.UnstructuredGrid(*vtk_mesh)
plotter = pyvista.Plotter()
plotter.add_mesh(grid, style="wireframe", color="r")
plotter.add_mesh(glyphs)
plotter.view_xy()
if not pyvista.OFF_SCREEN:
    plotter.show()
else:
    plotter.screenshot(outdir / "cylinder_fiber.png")

## Sheets: longitudinal direction

In [ ]:
assert geo.s0 is not None
topology, cell_types, geometry = dolfinx.plot.vtk_mesh(geo.s0.function_space)
values = np.zeros((geometry.shape[0], 3), dtype=np.float64)
values[:, : len(geo.s0)] = geo.s0.x.array.real.reshape((geometry.shape[0], len(geo.s0)))
function_grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)
function_grid["u"] = values
glyphs = function_grid.glyph(orient="u", factor=char_length)
grid = pyvista.UnstructuredGrid(*vtk_mesh)
plotter = pyvista.Plotter()
plotter.add_mesh(grid, style="wireframe", color="r")
plotter.add_mesh(glyphs)
plotter.view_xy()
if not pyvista.OFF_SCREEN:
    plotter.show()
else:
    plotter.screenshot(outdir / "cylinder_sheets.png")

## Sheet normal: radial direction

In [ ]:
assert geo.n0 is not None
topology, cell_types, geometry = dolfinx.plot.vtk_mesh(geo.n0.function_space)
values = np.zeros((geometry.shape[0], 3), dtype=np.float64)
values[:, : len(geo.n0)] = geo.n0.x.array.real.reshape((geometry.shape[0], len(geo.n0)))
function_grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)
function_grid["u"] = values
glyphs = function_grid.glyph(orient="u", factor=char_length)
grid = pyvista.UnstructuredGrid(*vtk_mesh)
plotter = pyvista.Plotter()
plotter.add_mesh(grid, style="wireframe", color="r")
plotter.add_mesh(glyphs)
plotter.view_xy()
if not pyvista.OFF_SCREEN:
    plotter.show()
else:
    plotter.screenshot(outdir / "cylinder_sheets_normal.png")

Next we transform the geometry to a `HeartGeometry` object

In [ ]:
geometry = pulse.HeartGeometry.from_cardiac_geometries(
    geo, metadata={"quadrature_degree": 6}
)

In [ ]:
# material_params = pulse.HolzapfelOgden.transversely_isotropic_parameters()
# material = pulse.HolzapfelOgden(f0=geo.f0, s0=geo.s0, **material_params)  # type: ignore
mu = pulse.Variable(dolfinx.fem.Constant(geometry.mesh, 10.0), "kPa")
material = pulse.NeoHookean(mu=mu)
print(material)

In [ ]:
active_model = pulse.active_model.Passive()

In [ ]:
comp_model = pulse.compressibility.Incompressible()
print(comp_model)

and assembles the `CardiacModel`

In [ ]:
model = pulse.CardiacModel(
    material=material,
    active=active_model,
    compressibility=comp_model,
)

In [ ]:
traction = pulse.Variable(
    dolfinx.fem.Constant(geometry.mesh, dolfinx.default_scalar_type(0.0)),
    "Pa",
)
robin_value_epi = pulse.Variable(
    dolfinx.fem.Constant(geometry.mesh, dolfinx.default_scalar_type(2e6)),
    "Pa / m",
)
robin_value_base = pulse.Variable(
    dolfinx.fem.Constant(geometry.mesh, dolfinx.default_scalar_type(8e6)),
    "Pa / m",
)
neumann = (
    pulse.NeumannBC(traction=traction, marker=geometry.markers["INSIDE"][0]),    
)
robin = (
    pulse.RobinBC(value=robin_value_base, marker=geometry.markers["TOP"][0]),
    pulse.RobinBC(value=robin_value_base, marker=geometry.markers["BOTTOM"][0]),
    pulse.RobinBC(value=robin_value_epi, marker=geometry.markers["OUTSIDE"][0]),
)
parameters = pulse.problem.StaticProblem.default_parameters()
parameters["mesh_unit"] = "m"

In [ ]:
# Next we set up the problem.
# bcs = pulse.BoundaryConditions(robin=robin, neumann=neumann, dirichlet=(dirichlet_bc,))
bcs = pulse.BoundaryConditions(robin=robin, neumann=neumann)
problem = pulse.problem.StaticProblem(
    model=model,
    geometry=geometry,
    bcs=bcs,
    parameters=parameters,
)

Set up variables for postprocessing

In [ ]:
u = dolfinx.fem.Function(problem.u_space)
p = dolfinx.fem.Function(problem.p_space)

# Make models for postprocessing
comp_post = pulse.compressibility.Incompressible()
comp_post.register(p)
material_post = pulse.NeoHookean(mu=mu)
active_post = pulse.active_model.Passive()
model_post = pulse.CardiacModel(
    material=material_post,
    active=active_post,
    compressibility=comp_post,
)

W = dolfinx.fem.functionspace(geometry.mesh, ("DG", 1))
W_tensor = dolfinx.fem.functionspace(geometry.mesh, ("DG", 1, (3, 3)))
I = ufl.Identity(3)
F = ufl.variable(ufl.grad(u) + I)
C = F.T * F
E = 0.5 * (C - I)
T = model_post.sigma(F)
Tdev = ufl.dev(T)
S = model_post.S(ufl.variable(C))

In [ ]:
# Fibers in current configuration
c = (F * geo.f0) / ufl.sqrt(ufl.inner(F * geo.f0, F * geo.f0))
l = (F * geo.s0) / ufl.sqrt(ufl.inner(F * geo.s0, F * geo.s0))
r = (F * geo.n0) / ufl.sqrt(ufl.inner(F * geo.n0, F * geo.n0))

In [ ]:
circ_stress_expr = dolfinx.fem.Expression(
    ufl.inner(T * c, c),
    W.element.interpolation_points,
)
circ_dev_stress_expr = dolfinx.fem.Expression(
    ufl.inner(Tdev * c, c),
    W.element.interpolation_points,
)
long_stress_expr = dolfinx.fem.Expression(
    ufl.inner(T * l, l),
    W.element.interpolation_points,
)
long_dev_stress_expr = dolfinx.fem.Expression(
    ufl.inner(Tdev * l, l),
    W.element.interpolation_points,
)
rad_stress_expr = dolfinx.fem.Expression(
    ufl.inner(T * r, r),
    W.element.interpolation_points,
)
rad_dev_stress_expr = dolfinx.fem.Expression(
    ufl.inner(Tdev * r, r),
    W.element.interpolation_points,
)
circ_strain_expr = dolfinx.fem.Expression(
    ufl.inner(E * geo.f0, geo.f0),
    W.element.interpolation_points,
)
long_strain_expr = dolfinx.fem.Expression(
    ufl.inner(E * geo.s0, geo.s0),
    W.element.interpolation_points,
)
rad_strain_expr = dolfinx.fem.Expression(
    ufl.inner(E * geo.n0, geo.n0),
    W.element.interpolation_points,
)
# Just interpolate p in the same space as the rest
p_expr = dolfinx.fem.Expression(
    p, W.element.interpolation_points,
)

Now we can solve the problem

In [ ]:
checkpoint_file = outdir / "checkpoint.bp"
if checkpoint_file.exists():
    timesteps = adios4dolfinx.read_timestamps(checkpoint_file, comm, function_name="u")
else:
    timesteps = []

In [ ]:
us = []
ps = []
pressures = np.linspace(0, target_pressure, 5)
for pressure in pressures:
    print(f"Solving for pressure {pressure} Pa")
    traction.assign(pressure)

    ui = dolfinx.fem.Function(problem.u_space, name="u")
    pi = dolfinx.fem.Function(problem.p_space, name="p")

    if pressure in timesteps:
        adios4dolfinx.read_function(
            filename=checkpoint_file, u=ui, time=pressure, name="u"
        )
        adios4dolfinx.read_function(
            filename=checkpoint_file, u=pi, time=pressure, name="p"
        )
    else:
        problem.solve()
        ui.x.array[:] = problem.u.x.array.copy()
        pi.x.array[:] = problem.p.x.array.copy()
        adios4dolfinx.write_function_on_input_mesh(
            filename=checkpoint_file, u=ui, time=pressure
        )
        adios4dolfinx.write_function_on_input_mesh(
            filename=checkpoint_file, u=pi, time=pressure
        )

    us.append(ui)
    ps.append(pi)

## Visualization of results

### Displacement

Define displacement magnitude

In [ ]:
V_mag = dolfinx.fem.functionspace(geo.mesh, ("Lagrange", 2))
magnitude = dolfinx.fem.Function(V_mag)
u_mag_expr = dolfinx.fem.Expression(
    ufl.sqrt(sum([u[i] ** 2 for i in range(len(u))])), V_mag.element.interpolation_points
)
magnitude.interpolate(u_mag_expr)

Create a gif showing the deformation

In [ ]:
plotter = pyvista.Plotter()
vmin = 0.0
vmax = 0.005
topology_p, cell_types_p, geometry_p = dolfinx.plot.vtk_mesh(u.function_space)
grid = pyvista.UnstructuredGrid(topology_p, cell_types_p, geometry_p)
grid["u"] = u.x.array.reshape((geometry_p.shape[0], 3))
actor_0 = plotter.add_mesh(grid, style="wireframe", color="k")
warped = grid.warp_by_vector("u", factor=1.0)
warped["mag"] = magnitude.x.array
warped.set_active_scalars("mag")
plotter.add_mesh(warped, show_edges=False, opacity=0.5, n_colors=10)
plotter.update_scalar_bar_range([vmin, vmax])
# Warp the mesh by the displacement vector to visualize deformation
#warped = grid.warp_by_vector("u", factor=1.0)
#actor_1 = p.add_mesh(warped, show_edges=False, color="red", opacity=0.5)
plotter.show_axes()
pressures = np.linspace(0, target_pressure, 5)
plotter.open_gif(outdir / 'displacement.gif', fps=2)

for i, ui in enumerate(us):
    u.x.array[:] = ui.x.array  
    magnitude.interpolate(u_mag_expr)
    warped_n = grid.warp_by_vector(factor=1)
    warped.points[:, :] = warped_n.points
    warped["mag"] = magnitude.x.array
    plotter.add_text(f"p = {pressures[i]} kPa", name="text")
    plotter.write_frame()
    plotter.remove_actor("text")
plotter.close()

In [ ]:
from IPython.display import Image
Image(outdir / 'displacement.gif')

### Stress and strain

In [ ]:
import pyvista as pv
import numpy as np
import dolfinx
import basix

# --- Configuration ---
pressures = np.linspace(0, 10.0, 5) 

# Field Configuration
field_expr = {
    "Circumferential Strain": circ_strain_expr,
    "Circumferential Stress": circ_stress_expr,
    "Circumferential Deviatoric Stress": circ_dev_stress_expr,
    "Longitudinal Strain": long_strain_expr,
    "Longitudinal Stress": long_stress_expr,
    "Longitudinal Deviatoric Stress": long_dev_stress_expr,
    "Radial Strain": rad_strain_expr,
    "Radial Stress": rad_stress_expr,
    "Radial Deviatoric Stress": rad_dev_stress_expr, 
    "p": p
}
funcs = {k: [] for k in field_expr.keys()}


circ_strain = dolfinx.fem.Function(W)
el_circ = circ_strain.function_space.ufl_element().basix_element
el = basix.ufl.element(family=el_circ.family, cell=el_circ.cell_type, 
                       degree=el_circ.degree, discontinuous=el_circ.discontinuous, shape=(3,))
V_u = dolfinx.fem.functionspace(geo.mesh, el)
u_int = dolfinx.fem.Function(V_u)

print("Pre-computing full meshes (Unclipped)...")
precomputed_meshes = {} 

for t_step, (ui, pi) in enumerate(zip(us, ps)):
    # Update Physics
    u.x.array[:] = ui.x.array
    u_int.interpolate(u)
    p.x.array[:] = pi.x.array

    # Create Grid
    topology_p, cell_types_p, geometry_p = dolfinx.plot.vtk_mesh(V_u)
    grid = pyvista.UnstructuredGrid(topology_p, cell_types_p, geometry_p)
    ref = grid
    grid["u"] = u_int.x.array.reshape((geometry_p.shape[0], 3))
    
    # Warp
    warped = grid.warp_by_vector("u", factor=1.0)
    
    # Store Fields
    for field_name, expr in field_expr.items():
        f = dolfinx.fem.Function(W)
        f.interpolate(expr)
        funcs[field_name].append(f)
        warped.point_data[field_name] = f.x.array.copy()

    # NOTE: We do NOT clip here anymore. We store the full mesh.
    precomputed_meshes[t_step] = warped

print(f"Done! Cached {len(precomputed_meshes)} steps.")

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import pyvista as pv

def plot_frame(t_idx, field_name, clip_mode, show_ref, crinkle, show_edges, stress_range, strain_range, cmap):
    # 1. Fetch the full mesh
    mesh = precomputed_meshes[t_idx]
    
    # 2. Apply Dynamic Clipping
    # This creates a temporary copy, leaving the original 'mesh' safe
    if clip_mode == "Horizontal":
        # Based on your original code: normal='x', invert=True, crinkle=True
        mesh_to_show = mesh.clip(normal="x", invert=True, crinkle=crinkle)
    elif clip_mode == "Vertical":
        # Based on your original code: normal='z', invert=True, crinkle=True
        mesh_to_show = mesh.clip(normal="z", invert=True, crinkle=crinkle)
    else:
        # Show full mesh
        mesh_to_show = mesh

    # 3. Setup Plotter
    p = pv.Plotter(notebook=True)

    if show_ref:
        actor_0 = p.add_mesh(grid, style="wireframe", color="k")

    if 'strain' in field_name.lower():
        clim = strain_range
    elif 'stress' in field_name.lower():
        clim = stress_range
    else:
        clim = stress_range # p
    
    p.add_mesh(
        mesh_to_show, 
        scalars=field_name, 
        cmap=cmap, 
        clim=clim, 
        show_edges=show_edges
    )
    
    # Update Text
    p.add_text(f"Pressure: {pressures[t_idx]:.2f} kPa", position='upper_left', font_size=10)
    p.add_text(f"Clip: {clip_mode}", position='upper_right', font_size=9, color='grey')
    p.show_axes()
    
    # 4. Show (using 'client' or 'static' if client fails)
    p.show(jupyter_backend='html')

# --- Widget Setup ---

slider = widgets.IntSlider(
    min=0, max=len(us)-1, 
    description="Time Step", 
    continuous_update=False
)

dropdown_field = widgets.Dropdown(
    options=list(field_expr.keys()), 
    value="Circumferential Strain", 
    description="Field"
)

# New Clipping Dropdown
dropdown_clip = widgets.Dropdown(
    options=["None", "Horizontal", "Vertical"],
    value="Horizontal",
    description="Clipping"
)

# 
show_ref_mesh = widgets.Checkbox(
    value=False,
    description='Show reference mesh',
)

crinkle_clip = widgets.Checkbox(
    value=False,
    description='Crinkle clip',
)

show_edges = widgets.Checkbox(
    value=False,
    description='Show edges',
)

stress_range = widgets.FloatRangeSlider(
    value=[0, 5000],
    min=-10_000,
    max=10_000,
    step=100,
    description='Stress range:',
    orientation='horizontal',
)
strain_range = widgets.FloatRangeSlider(
    value=[-0.1, 0.1],
    min=-0.5,
    max=0.5,
    step=0.005,
    description='Strain range:',
    orientation='horizontal',
)

cmap = widgets.Dropdown(
    options=["viridis", "plasma"],
    value="viridis",
    description="Colormap"
)

# Layout: Stack the dropdowns next to each other
ui = widgets.VBox([
    widgets.HBox([slider, crinkle_clip, show_edges]),
    widgets.HBox([dropdown_field, dropdown_clip, show_ref_mesh]),
    widgets.HBox([stress_range, strain_range]),
    widgets.HBox([cmap]),
])

# Bind inputs to function
out = widgets.interactive_output(plot_frame, {
    't_idx': slider, 
    'field_name': dropdown_field,
    'clip_mode': dropdown_clip,
    'show_ref': show_ref_mesh,
    'crinkle': crinkle_clip,
    'show_edges': show_edges,
    'stress_range': stress_range,
    'strain_range': strain_range,
    'cmap': cmap,
                  
})

display(ui, out)

In [ ]:
import scifem
import matplotlib.pyplot as plt

def plot_trace(field_name, r_range, num_points, theta, z_value):
    r_start, r_end = r_range
    eps = 1e-8
    r = np.linspace(r_start + eps, r_end - eps, num_points)
    z_value = height / 2
    x = r * np.sin(theta)
    y = r * np.cos(theta)
    z = height / 2 * np.ones_like(x)

    fs = funcs[field_name]
    points = np.vstack([x, y, z]).T
    
    fig, ax = plt.subplots()
    for i, p in enumerate(pressures):
        values = scifem.evaluate_function(fs[i], points).squeeze()
        valid_values = ~np.isinf(values)
        ax.plot(r[valid_values], values[valid_values], label=f"p={p:.1f} kPa")
    ax.legend()
    ax.set_ylabel(field_name)
    ax.set_xlabel("Radius")
    plt.show()


dropdown_field = widgets.Dropdown(
    options=list(field_expr.keys()), 
    value="Circumferential Strain", 
    description="Field"
)
r_range = widgets.FloatRangeSlider(
    value=[min(r_inner_x, r_inner_y), max(r_outer_x, r_outer_y)],
    min=min(r_inner_x,r_inner_y),
    max=max(r_outer_x,r_outer_y),
    step=(max(r_outer_x,r_outer_y) - min(r_inner_x,r_inner_y))/10,
    description='Radius:',
    orientation='horizontal',
)

z_value = widgets.FloatSlider(
    value=height/2,
    min=0,
    max=height,
    step=height / 10.0,
    description='Z:',
    orientation='horizontal',
)

theta_value = widgets.FloatSlider(
    value=0.0,
    min=0,
    max=2 * np.pi,
    step=np.pi / 10,
    description='Theta:',
    orientation='horizontal',
)

num_points = widgets.IntSlider(
    value=20,
    min=0,
    max=100,
    step=1,
    description='Num points:',
    orientation='horizontal',
)


# Layout: Stack the dropdowns next to each other
ui = widgets.VBox([
    widgets.HBox([dropdown_field, num_points]),
     widgets.HBox([r_range, z_value, theta_value]),
])

# Bind inputs to function
out = widgets.interactive_output(plot_trace, {
    'field_name': dropdown_field,
    'r_range': r_range, 
    'num_points': num_points, 
    'theta': theta_value, 
    'z_value': z_value,       
})

display(ui, out)

In [ ]:
def theta_interpolant(x):
    x1, x2, x3 = x
    return np.arctan(x2 / x1) + np.pi / 2

theta_func = dolfinx.fem.Function(W)
theta_func.interpolate(theta_interpolant)
vtk_mesh = dolfinx.plot.vtk_mesh(W)
grid = pyvista.UnstructuredGrid(*vtk_mesh)
grid.point_data["theta"] = theta_func.x.array
plotter = pyvista.Plotter()
plotter.add_mesh(grid, show_edges=True)
plotter.view_xy()
plotter.show_bounds()
if not pyvista.OFF_SCREEN:
    plotter.show()
else:
    plotter.screenshot(outdir / "cylinder_mesh.png")

In [ ]:
def plot_heatmap(
    i,
    field_name="Circumferential Deviatoric Stress",
    num_points_theta = 50,
    num_points_r = 50,
):
    r_start, r_end = (min(r_inner_x, r_inner_y), max(r_outer_x, r_outer_y))
    eps = 1e-8
    
    nr = num_points_r // 5
    ntheta = num_points_theta // 5
    theta = np.linspace(0, 2 * np.pi, num_points_theta)
    r = np.linspace(r_start + eps, r_end - eps, num_points_r)
    
    rv, thetav = np.meshgrid(r, theta)
    z_value = height / 2
    x = rv * np.sin(thetav)
    y = rv * np.cos(thetav)
    z = height / 2 * np.ones_like(x)
    
    fs = funcs[field_name]
    points = np.stack([x, y, z]).reshape(3, -1).T
    fig, ax = plt.subplots()
    
    values = scifem.evaluate_function(fs[i], points).squeeze().reshape(x.shape)
    im = ax.imshow(values)
    ax.set_title(f"p = {pressures[i]}")
    ax.set_xticks(range(0, len(r), nr))
    ax.set_xticklabels([f"{ri:.3f}" for ri in r[::nr]])
    ax.set_yticks(range(0, len(theta), ntheta))
    ax.set_yticklabels([f"{thetai:.2f}" for thetai in theta[::ntheta]])
    ax.set_xlabel("r")
    ax.set_ylabel("theta")
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(field_name)
    fig.tight_layout()
    plt.show()


slider = widgets.IntSlider(
    min=0, max=len(us)-1, 
    description="Time Step", 
    continuous_update=False
)
dropdown_field = widgets.Dropdown(
    options=list(field_expr.keys()), 
    value="Circumferential Strain", 
    description="Field"
)

num_points_theta = widgets.IntSlider(
    value=20,
    min=0,
    max=100,
    step=1,
    description='N (theta):',
    orientation='horizontal',
)
num_points_r = widgets.IntSlider(
    value=20,
    min=0,
    max=100,
    step=1,
    description='N (r):',
    orientation='horizontal',
)


# Layout: Stack the dropdowns next to each other
ui = widgets.VBox([
    widgets.HBox([slider, dropdown_field]),
     widgets.HBox([num_points_theta, num_points_r]),
])

# Bind inputs to function
out = widgets.interactive_output(plot_heatmap, {
    'field_name': dropdown_field,
    'num_points_r': num_points_r,
    'num_points_theta': num_points_theta,
    'i': slider,
})
display(ui, out)

## All outputs

In [ ]:
for field_name in field_expr.keys():
    print(field_name)
    plot_frame(
        len(us)-1, 
        field_name, 
        clip_mode="None", 
        show_ref=False, 
        crinkle=False, 
        show_edges=True, 
        stress_range=None, 
        strain_range=None, 
        cmap="viridis"
    )

In [ ]:
for field_name in field_expr.keys():
    print(field_name)
    plot_trace(field_name, r_range=(min(r_inner_x, r_inner_y), max(r_outer_x, r_outer_y)), num_points=20, theta=0, z_value=height/2)

In [ ]:
for field_name in field_expr.keys():
    print(field_name)
    plot_heatmap(len(us)-1, field_name=field_name)